In [ ]:
!pip show langchain

Name: langchain
Version: 1.3.18
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
!pip install -qU langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
!pip install -qU langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [ ]:
gemini_api_key = userdata.get('gemini_api_key')

In [ ]:
model = init_chat_model(
    model = "google_genai:gemini-2.5-flash",
    api_key = gemini_api_key,
)

Create skill demand tool

In [ ]:
!pip install -qU langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch
from pprint import pprint

In [ ]:
tavily_api_key = userdata.get('TAVILY_API_KEY')

In [ ]:
skill_demand_tool = TavilySearch(
   max_results = 5,
   topic = "general",
   search_depth = "advanced",
   tavily_api_key = tavily_api_key
)

Create job Search Tool

In [ ]:
rapid_api_key = userdata.get('RAPID_API_KEY')

In [ ]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPID_API_KEY')

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

Create an Agent

In [ ]:
from langchain.agents import create_agent

In [ ]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [ ]:
agent = create_agent(
    model = model,
    tools = [skill_demand_tool, search_jobs],
    system_prompt = system_prompt,
    checkpointer=checkpointer,
    debug = True
)

In [ ]:
config = {"configurable": {
    "thread_id": "1"
    }}

In [ ]:
user_query = "What's the demand for generative AI in the industry and show me related job openings in India"

In [ ]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_query
        }
    ]
} config = config)

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='513406e4-4555-4214-8a22-f241d57ef4ff')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"location": "India", "skill": "generative AI"}'}, '__gemini_function_call_thought_signatures__': {'ec16c6dd-6a5b-4c73-abe9-ee77e55c9791': 'CuUDARFNMg+OsdvYd79e8+J4jbcsQ8wkYaghDBoeWylf6kkEawLDgUv47gGDua4f5kwgiU0pag4ZyJISpl0ek2z82hG8pMlgQEYTI8gSlkIp08VS+QH4ceweeqx78i+vKVzKTNAzwqZp8953ee0LQ/gMcIQM2nZgnmq0+CZxmsET/ZyRod9RYM0TmQGBak+xZQUKGHi6vk02idqxUxY/nASSSIi2rUiqcTz00f6Ro3+l9StddCz0wwkJdPQ921hsOtyYtvlTYA91y8aHAgWT7oFgOgRyQeMEZa648QXPBNPNZu2mB2g5PF+OC5V4Hm8/dI7Xsa1K201TLPmTsO2VLwTqvCbnjqaTdpfeFzPGGpJ/rDvAyscmnF7oHK/mncj5Vi9rb9eQwFPOLfwFCwvwYfhGsmZqUu3orNpvlt40EctBQjcrnJmW+zI4WiLMGsqKKNOEqhUGPXKDHJpJe/Fq7uyRrFhC9KF/KUsWXnSXSVEEmlgThv6B

In [ ]:
print(response["messages"][-1].content[0]['text'])

Here's what I found regarding the demand for Generative AI in the industry and related job opportunities:

Generative AI Industry Demand, Salary Insights, and Career Trends:

  Demand and Growth:
    Organizations are heavily investing in AI-powered automation, rapidly expanding the job market for Generative AI, particularly in Data Science.
    The job market for Generative AI is expected to remain strong, with companies in finance, healthcare, SaaS, and enterprise tech growing their Generative AI teams.
    There is a current gap between the demand for Generative AI skills and the availability of professionals with those skills, especially in India, contributing to high salaries.
    The Generative AI market is projected to grow at a rapid pace, with a CAGR of 30.8% between 2026 and 2036.

  Key Career Opportunities:
    AI Data Scientist: Designs and trains Generative AI models.
    AI Research Scientist: Enhances core generative algorithms.
    AI-Augmented Data Analyst: Uses Gener